In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Hello World: 첫 번째 양자 Circuit

[Bell 상태](https://en.wikipedia.org/wiki/Bell_state)(얽힌 두 Qubit)를 만들고 세 가지 방법으로 실행해 봅니다:

1. **이상적 시뮬레이션** — 완벽한 결과, 계정 불필요
2. **노이즈 시뮬레이션** — 실제 장치를 시뮬레이션, 계정 불필요
3. **실제 양자 하드웨어** — [IBM Quantum 계정](https://janlahmann.github.io/Qiskit-documentation/#setting-up-your-ibm-quantum-account) 필요

## Circuit 만들기

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

## 옵션 1: 이상적 시뮬레이션 (계정 불필요)
`StatevectorSampler`를 사용합니다 — 노이즈 없이 완벽한 결과를 제공하는 로컬 시뮬레이터입니다.

In [ ]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts)

## 옵션 2: 노이즈 시뮬레이션 (계정 불필요)
`FakeManilaV2`를 사용합니다 — 노이즈 특성을 포함하여 실제 IBM 양자 장치를 모방하는 로컬 시뮬레이터입니다. Circuit은 먼저 Transpiler를 통해 장치의 게이트 집합과 Qubit 연결성에 맞게 변환(적응)되어야 합니다.

In [ ]:
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = FakeManilaV2()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## 옵션 3: 실제 양자 하드웨어
IBM Quantum 계정이 필요합니다. 자세한 내용은 [IBM Quantum 계정 설정하기](https://janlahmann.github.io/Qiskit-documentation/#setting-up-your-ibm-quantum-account)를 참고하세요.

이 Binder 세션에서 아직 자격 증명을 저장하지 않으셨다면, 먼저 다음을 실행하세요:

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token="<your-api-key>",
    instance="<your-crn>",
    overwrite=True
)

**참고:** 실제 하드웨어에서의 작업은 대기열 시간에 따라 시간이 걸릴 수 있습니다. 셀이 아직 실행 중이라면 [quantum.cloud.ibm.com/workloads](https://quantum.cloud.ibm.com/workloads?user=me)에서 작업 상태를 확인하고 결과를 볼 수 있습니다.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## 다음 단계는?
- **[튜토리얼](https://mybinder.org/v2/gh/JanLahmann/Qiskit-documentation/main?filepath=docs/tutorials)** — 알고리즘, 오류 완화, 변환 등에 관한 단계별 가이드
- **[강좌](https://mybinder.org/v2/gh/JanLahmann/Qiskit-documentation/main?filepath=learning/courses)** — 양자 기초부터 유틸리티 규모 컴퓨팅까지 체계적인 학습 경로
- **[로컬 테스트 모드](https://janlahmann.github.io/Qiskit-documentation/#no-token-use-local-testing-mode)** — IBM Quantum 계정 없이 대부분의 노트북 실행